In [1]:
from pynq.overlays.base import BaseOverlay
import time
import threading

base = BaseOverlay("base.bit")

In [2]:
%%microblaze base.PMODA

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
int write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

void clear_gpios(){
    for(int i = 0; i < 8; ++i){
        write_gpio(i,0);
    }
}

# Season color in HEX

In [3]:
spring_hex =   ["FBCEB1",
                "FFDAB3",
                "F88379",
                "FFF1C1",
                "FFFF99",
                "C4E1C1",
                "B3E3E2",
                "FFE28A",
                "FADADD",
                "FFB7A5",
                "D6C9FF",
                "F7E7CE",
                "FFBF00",
                "007FFF",
                "FF6F61",
                "4CBB17",
                "FF6347",
                "FFD700",
                "00A86B",
                "FF9966",
                "FFA500",
                "FF5C7A",
                "0F52BA",
                "40E0D0",
                "FF5E5B",
                "FFF700",
                "FF9900",
                "FF8C69",
                "7FFF00",
                "00CED1",
                "FF69B4",
                "FFFF00",
                "00FFFF",
                "FF6347",
                "DAA520",
                "A7FC00"]

summer_hex = ["B4CFEC",
              "FFD1DC",
              "B5EAD7",
              "C8A2C8",
              "C8A2C8",
              "FFF1C1",
              "E6E6FA",
              "B0E0E6",
              "98FF98",
              "F7CAC9",
              "87CEEB",
              "CCCCFF",
              "D4A5A5",
              "BCB88A",
              "6A5ACD",
              "E0B0FF",
              "C2B280",
              "483C32",
              "6B7B8C",
              "B39EB5",
              "8A9A5B",
              "B38481",
              "708090",
              "5E7D7E",
              "3B5998",
              "3E8E8E",
              "8E4585",
              "71797E",
              "C76B8A",
              "4A7C6F",
              "7366BD",
              "4F84C4",
              "6B4570",
              "5B8A8A",
              "5A6D7A",
              "9B3A62"]

autumn_hex =   ["D4AF37",
                "D8A48F",
                "A3AA8B",
                "C87C5A",
                "808000",
                "D2B1A3",
                "9B6A6C",
                "F5DEB3",
                "E2725B",
                "967969",
                "8A9A5B",
                "B08D57",
                "DAA520",
                "FF7518",
                "B87333",
                "E34234",
                "F4C430",
                "B7410E",
                "A0522D",
                "738678",
                "FFC512",
                "CD7F32",
                "6B8E23",
                "A0522D",
                "4A1C17",
                "556B2F",
                "4D5D53",
                "8A3324",
                "8B2500",
                "645452",
                "773F1A",
                "9B4722",
                "800000",
                "228B22",
                "5C3317",
                "996515"]

winter_hex =   ["007FFF",
                "990066",
                "800020",
                "36454F",
                "A5F2F3",
                "26619C",
                "FF00FF",
                "0F52BA",
                "8A2BE2",
                "C21E56",
                "708090",
                "CCCCFF",
                "0055FF",
                "FF69B4",
                "E0115F",
                "000000",
                "E5F9F9",
                "A020F0",
                "D9D9D6",
                "00FFFF",
                "D200D2",
                "3F00FF",
                "C0C0C0",
                "1C1C1C",
                "000080",
                "9B111E",
                "005F5F",
                "004B49",
                "4A0E0E",
                "580F41",
                "082567",
                "2B2B2B",
                "8B008B",
                "722F37",
                "2F4F4F",
                "004953"]

# Convert HEX to RGB

In [4]:
def hex_to_pwm_duty(hex_code, gamma=2.2):
    """
    correction in place for relative brightness
    """
    if len(hex_code) != 6:
        raise ValueError("Hex code must be 6 characters, e.g., 'A1B2C3'.")

    try:
        # hex -> 0-255
        r = int(hex_code[0:2], 16)
        g = int(hex_code[2:4], 16)
        b = int(hex_code[4:6], 16)

        # normalize
        r_norm = r / 255
        g_norm = g / 255
        b_norm = b / 255

        # correct to compensate for human preceived brightness
        r_gamma = r_norm ** gamma
        g_gamma = g_norm ** gamma
        b_gamma = b_norm ** gamma

        # 100 scale for duty cycle :D
        r_duty = int(r_gamma * 100)
        g_duty = int(g_gamma * 100)
        b_duty = int(b_gamma * 100)

        return (r_duty, g_duty, b_duty)

    except ValueError:
        raise ValueError("Invalid hex characters. Must be 0-9 or A-F.")

In [5]:
spring_rgb_pwm = []
for color in spring_hex:
    spring_rgb_pwm.append(hex_to_pwm_duty(color))

summer_rgb_pwm = []
for color in summer_hex:
    summer_rgb_pwm.append(hex_to_pwm_duty(color))

autumn_rgb_pwm = []
for color in autumn_hex:
    autumn_rgb_pwm.append(hex_to_pwm_duty(color))

winter_rgb_pwm = []
for color in winter_hex:
    winter_rgb_pwm.append(hex_to_pwm_duty(color))

In [ ]:
def color_cycler(season_list):
    """
    sliding window, one PMOD will start at 0 and 1 and the other will start at 2 and 3
    """
    idx0 = 0
    idx1 = 1
    n_colors = len(season_list)
    
    while not stop_event.is_set():
        # Update duty cycles for both LEDs
        r0,g0,b0 = season_list[idx0]
        duty[0]["R"] = r0
        duty[0]["G"] = g0
        duty[0]["B"] = b0

        r1,g1,b1 = season_list[idx1]
        duty[1]["R"] = r1
        duty[1]["G"] = g1
        duty[1]["B"] = b1

        idx0 = (idx0 + 1) % n_colors
        idx1 = (idx1 + 1) % n_colors

        # each color stays on for 1s
        time.sleep(1)

In [ ]:
def pwm_thread(color, num):
    pin = pins[num][color]
    while not stop_event.is_set():
        on_time = duty[num][color] / 100 / freq
        off_time = (1 - duty[num][color] / 100) / freq

        write_gpio(pin, 1)
        time.sleep(on_time)

        write_gpio(pin, 0)
        time.sleep(off_time)

# Main

In [10]:
freq = 50 

# LED pin mapping
pins = ({"R": 7, "G": 6, "B": 5}, {"R": 3, "G": 2, "B": 1})

stop_event = threading.Event()

# shared list for threads for duty cycles
duty = [{"R":0,"G":0,"B":0}, {"R":0,"G":0,"B":0}]

season_to_pwm = {
    "spring": spring_rgb_pwm,
    "summer": summer_rgb_pwm,
    "autumn": autumn_rgb_pwm,
    "winter": winter_rgb_pwm
}

season = "winter"
selected_season_list = season_to_pwm.get(season)

# --------------------------------------------------------------------
threads = []
for led_num in (0,1):
    for c in ["R","G","B"]:
        t = threading.Thread(target=pwm_thread, args=(c, led_num))
        t.start()
        threads.append(t)

cycler_thread = threading.Thread(target=color_cycler, args=(winter_rgb_pwm,))
cycler_thread.start()

print("PWM threads running. Press Enter to stop.")
input()

stop_event.set()
cycler_thread.join()
for t in threads:
    t.join()

print("All PWM threads stopped.")

PWM threads running. Press Enter to stop.

All PWM threads stopped.
